## Training: BERT + ResNet50 (max-fusion baseline)

This notebook trains a simple multimodal baseline:
- Text branch: BERT (`bert-base-uncased`) using the CLS token embedding
- Image branch: ResNet50 (ImageNet pretrained)
- Fusion: element-wise max between the two modality logits

### Inputs
- `CLEANDFPATH` / `FINAL_PATH`: cleaned CSVs produced by the download/validation pipeline
- `IMAGEDIR`: cached images named `{id}.jpg`

### Outputs
- Saves model weights to Drive (see the path in the training function)
- Prints basic evaluation metrics (accuracy/precision/recall) and optionally confusion matrix

### Notes
- This notebook contains some troubleshooting cells for missing images and retry-downloads.
- CrossEntropyLoss expects raw logits; do **not** apply softmax inside the model when training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
FINAL_PATH  = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df_final.csv"


In [ ]:
import torch

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as f
import torch.optim as optim

import torchvision
from torchvision.transforms import v2
from torchvision import models
from torchvision.models import resnet50, ResNet50_Weights
import torch.optim.lr_scheduler as lr_scheduler

import random
from PIL import Image
import matplotlib.pyplot as plt

## 1) Data paths + sanity checks

Load the cleaned CSV(s) and confirm the expected files exist in `IMAGEDIR`.
If you see many missing images, run the retry-download cell before training.


In [ ]:
import os
import pandas as pd
from datetime import datetime

CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"
IMAGEDIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images"

print("CLEANDFPATH:", CLEANDFPATH)
print("Exists:", os.path.exists(CLEANDFPATH))
print("Modified:", datetime.fromtimestamp(os.path.getmtime(CLEANDFPATH)))

df = pd.read_csv(CLEANDFPATH, sep=",")
print("Shape loaded in main notebook:", df.shape)

In [ ]:
df = pd.read_csv(FINAL_PATH, sep=",")
print(df.shape)

In [ ]:
all_files = set(os.listdir(IMAGEDIR))
print("Total image files visible:", len(all_files))

In [ ]:
missing = [
    str(row["id"]) for _, row in df.iterrows()
    if f"{row['id']}.jpg" not in all_files
]
print("Missing images:", len(missing))

In [ ]:
from urllib import request, error
from PIL import Image
df = pd.read_csv(CLEANDFPATH)
os.makedirs(IMAGEDIR, exist_ok=True)

all_files = set(os.listdir(IMAGEDIR))
missing_df = df[~df["id"].astype(str).apply(lambda x: f"{x}.jpg" in all_files)].copy().reset_index(drop=True)

print("Missing before retry:", len(missing_df))

success_ids = []
failed_info = []

for i, row in missing_df.iterrows():
    img_id = str(row["id"])
    image_url = row["image_url"]
    img_path = os.path.join(IMAGEDIR, f"{img_id}.jpg")

    ok = False
    last_err = None

    for attempt in range(3):
        try:
            req = request.Request(
                image_url,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            with request.urlopen(req, timeout=20) as resp:
                img_bytes = resp.read()

            with open(img_path, "wb") as f:
                f.write(img_bytes)

            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img.verify()

            ok = True
            success_ids.append(img_id)
            break

        except Exception as e:
            last_err = str(e)
            if os.path.exists(img_path):
                try:
                    os.remove(img_path)
                except:
                    pass
            time.sleep(2 * (attempt + 1))

    if not ok:
        failed_info.append((img_id, image_url, last_err))

    if (i + 1) % 25 == 0 or (i + 1) == len(missing_df):
        print(f"{i+1}/{len(missing_df)} processed | success={len(success_ids)} | failed={len(failed_info)}")

failed_df = pd.DataFrame(failed_info, columns=["id", "image_url", "error"])
failed_df.to_csv("/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/retry_failed_504.csv", index=False)

print("Retry success:", len(success_ids))
print("Still failed:", len(failed_info))

In [ ]:
new_size = (256, 256)

for img_id in success_ids:
    image_path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + str(img_id) + ".jpg"
    
    image = Image.open(image_path).convert("RGB")

    # Resize the image using PyTorch's torchvision.transforms
    resize_transform = v2.Resize(new_size)
    resized_image = resize_transform(image)
    resized_image.save(image_path)

## 2) Text preprocessing (BERT tokenization)

We tokenize each title to a fixed length (`max_length=80`).
The function returns tensors you can feed into `BertModel`.


In [ ]:
%%capture


! pip install bert-serving-server  # server-side
! pip install bert-serving-client  # client-side
! pip install torch transformers

import torch
from transformers import BertModel, BertTokenizer

# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name, output_hidden_states = True)

# Put the model in evaluation mode, which turns off dropout regularization which is used in training.
bert_model.eval()

In [ ]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        return_tensors="pt",
        max_length=80,
        truncation=True,
        padding="max_length",
        return_attention_mask=True
    )

    return inputs["input_ids"].squeeze(0), inputs["attention_mask"].squeeze(0)

# test
text = "This is an example Reddit submission title."
input_ids, attention_mask = get_bert_embedding(text)
print(input_ids.shape)        # torch.Size([80])
print(attention_mask.shape)   # torch.Size([80])

In [ ]:
from sklearn.model_selection import train_test_split
RANDOM_STATE=42
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df["6_way_label"],random_state=RANDOM_STATE)
df_test, df_val = train_test_split(df_test, test_size=0.5, stratify=df_test["6_way_label"], random_state=RANDOM_STATE)

## 3) Dataset + loaders

Each sample returns:
- tokenized text (`input_ids`, `attention_mask`)
- label
- normalized image tensor (ResNet/ImageNet normalization)


In [ ]:
class FakedditDataset(Dataset):
    def __init__(self, df, text_field="clean_title", label_field="6_way_label", image_id="id"):
        self.df = df.reset_index(drop=True)
        self.text_field = text_field
        self.label_field = label_field
        self.image_id = image_id

        self.img_size = 256
        # Using the pre-calculated ImageNet mean and std values for normalization
        self.mean, self.std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        self.transform_func = v2.Compose(
                [   v2.Resize(256),
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),
                    v2.Normalize(self.mean, self.std)
                    ])

    def __getitem__(self, index):
        text = str(self.df.at[index, self.text_field])
        label = self.df.at[index, self.label_field]
        img_path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + self.df.at[index, self.image_id] + ".jpg"

        image = Image.open(img_path)
        img = self.transform_func(image)

        input_ids, attention_mask = get_bert_embedding(text)

        return input_ids, attention_mask, label, img

    def __len__(self):
        return self.df.shape[0]

In [ ]:
train_dataset = FakedditDataset(df_train)
test_dataset = FakedditDataset(df_test)
val_dataset = FakedditDataset(df_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(len(train_loader))

# Verifying dataset was created accurately
input_ids, attention_mask, label, img = next(iter(train_loader))
print(input_ids.shape, attention_mask.shape, label.shape, img.shape)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
labels = df_train['6_way_label'].to_numpy()
type(labels)

## 4) Model + training loop

- ResNet50 is initialized with ImageNet weights.
- BERT is initialized from `bert-base-uncased`.
- Each branch produces logits over the 6 classes; fusion uses `torch.max` (a simple, strong baseline).
- We use class-weighted CrossEntropy to reduce bias toward frequent classes.


In [ ]:
# Define loss function and optimizer
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
# Assuming 'labels' is a list of all labels in the dataset
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Define the loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, min_lr=1e-6, factor=0.5, patience=1) #, verbose=True
num_epochs = 20

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    early_stopping = EarlyStopping(patience=5, verbose=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for step, (input_ids, attention_mask, label, img) in enumerate(train_loader, start=1):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            label = label.to(device)
            img = img.to(device)

            optimizer.zero_grad()
            outputs = model(img, input_ids, attention_mask)
            loss = criterion(outputs, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * img.size(0)

            if step % 50 == 0 or step == len(train_loader):
                print(f"Epoch {epoch+1}/{num_epochs} | Step {step}/{len(train_loader)} | Batch Loss: {loss.item():.4f}")

        model.eval()
        val_loss = 0.0
        correct_preds = 0

        with torch.no_grad():
            for input_ids, attention_mask, label, img in val_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                label = label.to(device)
                img = img.to(device)

                outputs = model(img, input_ids, attention_mask)
                loss = criterion(outputs, label)

                val_loss += loss.item() * img.size(0)
                _, preds = torch.max(outputs, 1)
                correct_preds += torch.sum(preds == label)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss = val_loss / len(val_loader.dataset)
        accuracy = correct_preds.double() / len(val_loader.dataset)

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} done | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {accuracy:.4f}")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break

    torch.save(model.state_dict(), "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/m1.pth")

In [ ]:
from sklearn.metrics import precision_score, recall_score

def evaluate_model(model, test_loader, criterion):
    model.eval()
    val_losses = []
    correct_preds = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, attention_mask, label, img in test_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            label = label.to(device)
            img = img.to(device)

            outputs = model(
                  image = img,
                  text_input_ids = input_ids,
                  text_attention_mask = attention_mask
            )

            # Final Softmax layer returns class predictions per sample in batch
            # Highest probability value resembles class prediction and is assigned to preds variable
            _, preds = torch.max(outputs, dim=1)
            print(outputs)

            # Loss is calculated by applying Cross Entropy Loss
            val_loss = criterion(outputs, label)

            # Counting correct model predictions and incrementing correct prediction count
            correct_preds += torch.sum(preds == label)
            print(preds, label)

            # Appending current loss per batch to list of losses per epoch
            val_losses.append(val_loss.item())
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            

    accuracy = float((correct_preds.double() / len(df_test)) * 100)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    print("\nAccuracy: ", accuracy)
    print("Precision: ", precision)
    print("Recall: ", recall)

In [ ]:
class BERTResNetClassifier(nn.Module):
    def __init__(self, num_classes=6):

        super(BERTResNetClassifier, self).__init__()

        self.num_classes = num_classes

        # Image branch: ResNet50 pretrained on ImageNet.
        # We keep the original `resnet50(...)->1000 logits` head and map it to our labels.
        self.image_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.fc_image = nn.Linear(in_features=1000, out_features=num_classes, bias=True)

        # Text branch: BERT pooled CLS embedding → logits.
        self.text_model = BertModel.from_pretrained("bert-base-uncased")
        self.fc_text = nn.Linear(
            in_features=self.text_model.config.hidden_size,
            out_features=num_classes,
            bias=True
        )

        # Regularization shared by both branches.
        self.drop = nn.Dropout(p=0.3)

        # Note: do NOT apply softmax here when using CrossEntropyLoss.

    def forward(self, image, text_input_ids, text_attention_mask,):
        # Image logits
        x_img = self.image_model(image)
        x_img = self.drop(x_img)
        x_img = self.fc_image(x_img)

        # Text logits (CLS token)
        x_text_last_hidden_states = self.text_model(
            input_ids=text_input_ids,
            attention_mask=text_attention_mask,
            return_dict=False
        )
        x_text_pooled_output = x_text_last_hidden_states[0][:, 0, :]
        x_text_pooled_output = self.drop(x_text_pooled_output)
        x_text = self.fc_text(x_text_pooled_output)

        # Fusion: take the stronger logit per class across modalities.
        x = torch.max(x_text, x_img)
        return x

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BERTResNetClassifier(num_classes=6).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, min_lr=1e-6, factor=0.5, patience=1)
num_epochs = 20

train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs)
evaluate_model(model, test_loader, criterion)

In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from torch.utils.data import DataLoader
import numpy as np

# # 1) Reload dataframe
# df = pd.read_csv(FINAL_PATH)   # or CLEANDFPATH if that is the one you trained on

# # 2) Recreate the exact same split
# RANDOM_STATE = 42
# df_train, df_test = train_test_split(
#     df,
#     test_size=0.2,
#     stratify=df["6_way_label"],
#     random_state=RANDOM_STATE
# )
# df_test, df_val = train_test_split(
#     df_test,
#     test_size=0.5,
#     stratify=df_test["6_way_label"],
#     random_state=RANDOM_STATE
# )

# # 3) Recreate test dataset/loader
# test_dataset = FakedditDataset(df_test)
# test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 4) Rebuild model and load saved weights
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BERTResNetClassifier(num_classes=6).to(device)
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/m1.pth",
    map_location=device
))
print("starting")
model.eval()
print("next phase")
# 5) Predict on test set
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, attention_mask, labels, imgs in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        imgs = imgs.to(device)

        outputs = model(imgs, input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 6) Metrics
cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, digits=4)
per_class_f1 = f1_score(all_labels, all_preds, average=None)

print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n")
print(report)

print("Per-class F1:")
for i, score in enumerate(per_class_f1):
    print(f"Class {i}: {score:.4f}")

print("\nPer-class accuracy:")
for cls in sorted(np.unique(all_labels)):
    idx = all_labels == cls
    acc = (all_preds[idx] == all_labels[idx]).mean()
    print(f"Class {cls}: {acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:

cm = confusion_matrix(all_labels, all_preds)
pairs = []

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], i, j))

pairs = sorted(pairs, reverse=True)

for count, true_cls, pred_cls in pairs[:10]:
    print(f"True class {true_cls} predicted as class {pred_cls}: {count}")